# Testes de persistência — Biblioteca
Java + JPA/Hibernate + H2. Execute `mvn clean package dependency:copy-dependencies -DincludeScope=runtime` na raiz e abra este notebook a partir de `notebooks/`, com o kernel Java (IJava). Reinicie o kernel antes de executar tudo novamente. Cada execução cria um banco temporário novo; nenhum banco existente é apagado.

In [1]:
%classpath add jar ../target/biblioteca-1.0.0.jar
%classpath add jar ../target/dependency/*.jar

In [2]:
import br.edu.biblioteca.*;
import java.time.LocalDate;
import java.nio.file.Files;
import jakarta.persistence.PersistenceException;
void verificar(boolean condicao, String descricao) {
    if (!condicao) throw new AssertionError(descricao);
    System.out.println("OK: " + descricao);
}
String arquivo = Files.createTempDirectory("biblioteca-jpa-").resolve("biblioteca").toString();
Database db = new Database(arquivo);
BibliotecaService servico = new BibliotecaService(db);
System.out.println("Banco de teste: " + arquivo);

Banco de teste: /tmp/biblioteca-jpa-6266383253211783245/biblioteca


## Cadastro e relação 1:1
O leitor pode ter zero ou uma carteirinha; cada carteirinha pertence a um único leitor.

In [3]:
Leitor leitor = new Leitor();
leitor.setNome("Bárbara"); leitor.setEmail("barbara@example.com");
db.repository(Leitor.class).save(leitor);
Carteirinha cartao = new Carteirinha();
cartao.setLeitor(leitor); cartao.setNumero("C001"); cartao.setEmitidaEm(LocalDate.parse("2026-09-09"));
db.repository(Carteirinha.class).save(cartao);
verificar(db.repository(Carteirinha.class).findById(cartao.getId()).getLeitor().getNome().equals("Bárbara"), "Carteirinha recupera seu leitor");
Carteirinha duplicada = new Carteirinha();
duplicada.setLeitor(leitor); duplicada.setNumero("C002"); duplicada.setEmitidaEm(LocalDate.parse("2026-09-09"));
try { db.repository(Carteirinha.class).save(duplicada); throw new AssertionError("Aceitou segunda carteirinha"); }
catch (PersistenceException esperado) { System.out.println("OK: segunda carteirinha rejeitada"); }

OK: Carteirinha recupera seu leitor


OK: segunda carteirinha rejeitada


## Relações 1:N e N:M
Dois livros e dois autores demonstram a autoria nos dois sentidos; um livro possui dois exemplares.

In [4]:
Livro livro = new Livro(); livro.setTitulo("Algoritmos em conjunto"); livro.setIsbn("9780000000001");
db.repository(Livro.class).save(livro);
Livro outroLivro = new Livro(); outroLivro.setTitulo("Dados em conjunto"); outroLivro.setIsbn("9780000000002");
db.repository(Livro.class).save(outroLivro);
Autor ana = new Autor(); ana.setNome("Ana Exemplo"); db.repository(Autor.class).save(ana);
Autor bruno = new Autor(); bruno.setNome("Bruno Exemplo"); db.repository(Autor.class).save(bruno);
for (Livro obra : java.util.List.of(livro, outroLivro)) {
    for (Autor autor : java.util.List.of(ana, bruno)) {
        LivroAutor autoria = new LivroAutor(); autoria.setLivro(obra); autoria.setAutor(autor);
        db.repository(LivroAutor.class).save(autoria);
    }
}
Exemplar exemplar = new Exemplar(); exemplar.setLivro(livro); exemplar.setCodigo("EX001"); db.repository(Exemplar.class).save(exemplar);
Exemplar copia = new Exemplar(); copia.setLivro(livro); copia.setCodigo("EX002"); db.repository(Exemplar.class).save(copia);
verificar(db.repository(Exemplar.class).findBy("livro.id", livro.getId()).size() == 2, "Livro possui dois exemplares");
verificar(db.repository(LivroAutor.class).findBy("livro.id", livro.getId()).size() == 2, "Livro possui dois autores");
verificar(db.repository(LivroAutor.class).findBy("autor.id", ana.getId()).size() == 2, "Autora participa de dois livros");

OK: Livro possui dois exemplares


OK: Livro possui dois autores


OK: Autora participa de dois livros


## Empréstimo, restrições e devolução
O mesmo exemplar só fica disponível para um novo empréstimo após a devolução.

In [5]:
Emprestimo emprestimo = servico.emprestar(leitor, exemplar, LocalDate.of(2026,9,9), LocalDate.of(2026,9,16));
try { servico.emprestar(leitor, exemplar, LocalDate.of(2026,9,9), LocalDate.of(2026,9,16)); throw new AssertionError("Aceitou empréstimo duplicado"); }
catch (PersistenceException esperado) { System.out.println("OK: exemplar já emprestado foi bloqueado"); }
servico.devolver(emprestimo.getId(), LocalDate.of(2026,9,12));
verificar(db.repository(Emprestimo.class).findById(emprestimo.getId()).getDevolvidoEm().equals(LocalDate.parse("2026-09-12")), "Devolução persistida");
servico.emprestar(leitor, exemplar, LocalDate.of(2026,9,13), LocalDate.of(2026,9,20));
verificar(db.repository(Emprestimo.class).findBy("exemplar.id", exemplar.getId()).size() == 2, "Histórico preserva dois empréstimos");

OK: exemplar já emprestado foi bloqueado


OK: Devolução persistida


OK: Histórico preserva dois empréstimos


## Atualização, exclusão e integridade referencial

In [6]:
leitor.setNome("Bárbara Nogueira"); db.repository(Leitor.class).update(leitor);
verificar(db.repository(Leitor.class).findById(leitor.getId()).getNome().equals("Bárbara Nogueira"), "Nome atualizado");
Autor temporario = new Autor(); temporario.setNome("Cadastro temporário"); db.repository(Autor.class).save(temporario);
db.repository(Autor.class).delete(temporario);
verificar(db.repository(Autor.class).findById(temporario.getId()) == null, "Autor sem vínculos excluído");
try { db.repository(Leitor.class).delete(leitor); throw new AssertionError("Excluiu leitor com vínculos"); }
catch (PersistenceException esperado) { System.out.println("OK: exclusão de leitor vinculado bloqueada"); }

OK: Nome atualizado


OK: Autor sem vínculos excluído


OK: exclusão de leitor vinculado bloqueada


## Transações JPA e rollback
Uma falha depois do flush deve desfazer a inserção inteira.

In [7]:
long autoresAntes = db.repository(Autor.class).count();
try {
    db.transaction(em -> {
        Autor autor = new Autor(); autor.setNome("Cadastro revertido");
        em.persist(autor); em.flush();
        throw new IllegalStateException("Falha simulada");
    });
    throw new AssertionError("A operação deveria falhar");
} catch (IllegalStateException esperado) {
    verificar("Falha simulada".equals(esperado.getMessage()), "Falha de teste capturada");
}
verificar(db.repository(Autor.class).count() == autoresAntes, "Rollback desfez a inserção");
int quantidadeExemplares = db.transaction(em -> em.find(Livro.class, livro.getId()).getExemplares().size());
verificar(quantidadeExemplares == 2, "Coleção OneToMany consultada dentro da transação");

OK: Falha de teste capturada


OK: Rollback desfez a inserção


OK: Coleção OneToMany consultada dentro da transação


## Persistência após fechar e reabrir
Esta consulta usa outra conexão, comprovando que os dados estão no arquivo H2.

In [8]:
int leitorId = leitor.getId();
db.close();
db = new Database(arquivo);
verificar(db.repository(Leitor.class).findById(leitorId).getNome().equals("Bárbara Nogueira"), "Dados preservados após reabrir o banco");
verificar(db.repository(Emprestimo.class).count() == 2, "Histórico preservado em disco");
db.close();
System.out.println("Testes concluídos. Banco disponível em: " + arquivo);

OK: Dados preservados após reabrir o banco


OK: Histórico preservado em disco


Testes concluídos. Banco disponível em: /tmp/biblioteca-jpa-6266383253211783245/biblioteca
